<a href="https://colab.research.google.com/github/KuchkorovBoburbek/computer_vision/blob/main/menu_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
print("Menu dedector")

Menu dedector


#  Menu detector loyihasi bu taomlar rasmlari orqali train qilib malum taomlarni taniydigan model

In [38]:
#---------------------
# Import Libraries
#---------------------
import torch
import torch.nn as nn
from google.colab import drive
import torchvision.transforms as transforms

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

from torchvision.models import mobilenet_v2



In [39]:
drive.mount('/content/drive') # drive ga ulanamiz

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
# Define dataset path
DATASET_PATH = '/content/drive/MyDrive/Food_101_dataset'


CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping  |  class consolidation
    'kebeb': 'kebeb',
    'pilav': 'pilaf',
    'ice_cream': 'dessert'     # label grouping  |  class consolidation
}





CLASSES = ['hamburger', 'dessert', 'hot_dog', 'kebeb', 'pilaf' ]
CLASS_TO_INDEX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES) # 5 ta

print(f'Classes: {CLASSES}')
print(f'Class to index: {CLASS_TO_INDEX}')
print(f'Number of classes: {NUM_CLASSES}')

# bu bizning ishlarimiz ketmaketligini belgilaydi
transform = transforms.Compose([
    transforms.Resize((224, 224)) , # bu rasmlarimiz sizeni o'zgartiradi
    transforms.ToTensor(), # tensorga o'giramiz(raqamlarga)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# Tensorga o'girgach u flot korinishiga otadi 0.0 ~ 1.0
# H, W, C => C, H, W  | => channel : RGB
# Normalize => hamma channellarni qayta sozlab beradi
# Normalize : pixel = (pixel-mean)/std



print("transform:", transform)


Classes: ['hamburger', 'dessert', 'hot_dog', 'kebeb', 'pilaf']
Class to index: {'hamburger': 0, 'dessert': 1, 'hot_dog': 2, 'kebeb': 3, 'pilaf': 4}
Number of classes: 5
transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [41]:
# -------------------------
# Custom Dataset Class
# -------------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
# umumiy filelarni o'qish vaqtida data uzunligini olib beradi
    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)
#
    def __getitem__(self, idx):
        img_path = self.images[idx]
        print('image_path', img_path)

        label = self.labels[idx]
        print('label', label)
# agar rasmda RGB bolmasa convert qilamiz
        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}") # rasm filelarda kamchilik bolsa skip qilish
            return self.__getitem__((idx + 1) % len(self.images)) # Bitta keyingisini

        if self.transform:
            image = self.transform(image)

        return image, label

In [42]:
# -------------------------
# Gather and Split Data
# -------------------------

all_images = [] # created empty list

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)  # /content/drive/MyDrive/food_101_dataset/hamburber
    print('class_path:', class_path)

    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img) # /content/drive/MyDrive/food_101_dataset/hamburber/233.jpg
            all_images.append((full_path, CLASS_TO_INDEX[mapped_class]))

np.random.shuffle(all_images)

split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))

img, lbl = dataset[0]

class_path: /content/drive/MyDrive/Food_101_dataset/hamburger
class_path: /content/drive/MyDrive/Food_101_dataset/hot_dog
class_path: /content/drive/MyDrive/Food_101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/Food_101_dataset/kebeb
class_path: /content/drive/MyDrive/Food_101_dataset/pilav
class_path: /content/drive/MyDrive/Food_101_dataset/ice_cream
images_length 3224
3224
image_path /content/drive/MyDrive/Food_101_dataset/chocolate_cake/3761798.jpg
label 1


In [43]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [44]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2) # thread 2ta
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)


images_length 3224
images_length 3224


In [45]:
# pretrained model : bu mavjud modelni olib ishlatish
model = mobilenet_v2(weights='IMAGENET1K_V1') #pretrained model | CNN | 1000dan ortiq class | mln ortiq rasm orqali train
print("model.classifier[1]:", model.classifier[0])
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
print("model.classifier[2]:", model.classifier[0])

model.classifier[1]: Dropout(p=0.2, inplace=False)
model.classifier[2]: Dropout(p=0.2, inplace=False)


In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("devive", device)
model = model.to(device)


devive cuda


In [47]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
torch.backends.cudnn.benchmark = True

In [49]:
# -------------------------
# Training Loop
# -------------------------
NUM_EPOCHS = 2
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
      images, labels = images.to(device), labels.to(device)
      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      running_loss += loss.item()

      # Validation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_acc = 100 * correct / total #Calculate Validation Accuracy

print(
    f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
    f"Loss: {running_loss/len(train_loader):.4f}, "
    f"Val Accuracy: {val_acc:.2f}%"
)

if val_acc > best_accuracy:
    best_accuracy = val_acc
    torch.save(model.state_dict(), "/content/menu_detector.pth")
    print("Saved new best model!")


Streaming output truncated to the last 5000 lines.
/content/drive/MyDrive/Food_101_dataset/chocolate_cake/1306649.jpg
labellabel 2 
1
image_path /content/drive/MyDrive/Food_101_dataset/chocolate_cake/1600861.jpgimage_path
 label/content/drive/MyDrive/Food_101_dataset/hot_dog/1186993.jpg
 1label
 2
image_path /content/drive/MyDrive/Food_101_dataset/hot_dog/171875.jpg
labelimage_path  2/content/drive/MyDrive/Food_101_dataset/hamburger/2668437.jpg

label 0
image_path image_path/content/drive/MyDrive/Food_101_dataset/chocolate_cake/3117730.jpg /content/drive/MyDrive/Food_101_dataset/hamburger/3583066.jpg

label label 10

image_path image_path/content/drive/MyDrive/Food_101_dataset/ice_cream/697142.jpg 
/content/drive/MyDrive/Food_101_dataset/chocolate_cake/3229898.jpglabel
 label 11

image_path /content/drive/MyDrive/Food_101_dataset/chocolate_cake/1474255.jpgimage_path
 /content/drive/MyDrive/Food_101_dataset/hot_dog/76904.jpglabel
 label1 
2
image_path /content/drive/MyDrive/Food_101_dat